In [1]:
!pip --quiet install ollama
!pip --quiet install pydantic


[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: pip install --upgrade pip


In [1]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel, Field # type: ignore
import json

## Marker Gene Extraction

In [76]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )
    tissue_source: Optional[str] = Field(
        None, 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: Optional[str] = Field(
        None, 
        description="The species from which this marker gene was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )
    tissue_source: str = Field(
        ..., 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: str = Field(
        ..., 
        description="The species from which this marker gene was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

# class MarkerGene(BaseModel):
#     marker_gene_name: Optional[str] = None
#     cell_name: Optional[str] = None
#     tissue_source: Optional[str] = None

# class MarkerGeneList(BaseModel):
#     genes: list[MarkerGene]

# class MarkerGeneStrict(BaseModel):
#     marker_gene_name: str
#     cell_name: str
#     tissue_source: str

# class MarkerGeneListStrict(BaseModel):
#     genes: list[MarkerGeneStrict]

In [73]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()["genes"]

In [74]:
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).
- The **tissue source** where this marker gene was identified (tissue_source).
- The **species** from which the marker gene was studied (species).

If any information is missing or ambiguous, provide the most specific details available. If none of the information is available, the field can be set to null.

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "marker_gene_name": "GeneX",
            "cell_type_name": "Neuron",
            "tissue_source": "Brain",
            "species": "Homo sapiens"
        },
        ...
    ]
}
"""
user_content = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), 
identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)
"""

system_prompt = "You are a genomics researcher trying to discover different cell types and what genes they express, aka find the marker genes within the given sentence."

In [ ]:
models = [
    "llama3.2",
    "deepseek-r1"
]

In [78]:
genes = extract_genes(user_content, system_prompt, MarkerGeneList, model="llama3.2")
print(len(genes))
print(json.dumps(genes, indent=4))

4
[
    {
        "marker_gene_name": null,
        "cell_type_name": "PG cells",
        "tissue_source": null,
        "species": "human"
    },
    {
        "marker_gene_name": null,
        "cell_type_name": "ING cells",
        "tissue_source": null,
        "species": "human"
    },
    {
        "marker_gene_name": null,
        "cell_type_name": "Male epithelial population",
        "tissue_source": null,
        "species": "human"
    },
    {
        "marker_gene_name": null,
        "cell_type_name": "Female epithelial population",
        "tissue_source": null,
        "species": "human"
    }
]


In [75]:
genes = extract_genes(user_content, system_prompt, MarkerGeneListStrict, model="llama3.2")
print(len(genes))
print(json.dumps(genes, indent=4))

2
[
    {
        "marker_gene_name": "Dcdc2a",
        "cell_name": "male epithelial population",
        "tissue_source": "human WAT"
    },
    {
        "marker_gene_name": "Erbb4",
        "cell_name": "female epithelial population",
        "tissue_source": "human WAT"
    }
]


In [79]:
genes = extract_genes(user_content, system_prompt, MarkerGeneList, model="deepseek-r1")
print(len(genes))
print(json.dumps(genes, indent=4))

2
[
    {
        "marker_gene_name": null,
        "cell_type_name": "Epithelial",
        "tissue_source": null,
        "species": "Human"
    },
    {
        "marker_gene_name": null,
        "cell_type_name": "WAT",
        "tissue_source": null,
        "species": "Human"
    }
]


In [80]:
genes = extract_genes(user_content, system_prompt, MarkerGeneListStrict, model="deepseek-r1")
print(len(genes))
print(json.dumps(genes, indent=4))

2
[
    {
        "marker_gene_name": "Dcdc2a",
        "cell_type_name": "male epithelial",
        "tissue_source": "human WAT",
        "species": "mouse"
    },
    {
        "marker_gene_name": "Erbb4",
        "cell_type_name": "female epithelial",
        "tissue_source": "human WAT",
        "species": "mouse"
    }
]


# Notes
- Benchmark positive examples and negative examples separately (in each case try out strict and permissive data model)
- Benchmark multiple models
- Group data source data by number of cell type marker gene pairs (i.e. for the same source_rationale, select all evidence derived from it to benchmark)
- Quantify llm extraction capability
    - how well the data was extracted from the input quantify errors for each key/value pair compared to the input text (use the function in analysis/human_vs_human/validate_evidence.ipynb)
    - how much hallucination, i.e. data in extracted but not in input
- Quantify extraction of marker gene to ground truth
    - first ensure the extracted data matches the text verbatim (modify if necessary)
    - second compare the (possibly corrected) extracted data, celltype and gene VVD
    - what other comparisons can we make?